# 03 — Baseline Model
Task 4: tomato_clean.csv → features → time-based split → baseline Linear Regression → metrics → saved model.

## 1. Load the cleaned data

In [ ]:
import pandas as pd
import numpy as np

df = pd.read_csv("../data/processed/tomato_clean.csv")

print("Dataset shape:", df.shape)

df.head()

## 2. Check the data

In [ ]:
print(df.info())

In [ ]:
print(df.isnull().sum())

In [ ]:
print(df.columns.tolist())

## 3. Convert date
Even though dates were handled in Task 3, convert explicitly here since this notebook does time-based splitting.

In [ ]:
df["date"] = pd.to_datetime(df["date"])

## 4. Sort chronologically
Critical for agricultural price prediction — the model must never train on future data to predict the past.

In [ ]:
df = df.sort_values("date").reset_index(drop=True)

## 5. Define the target

In [ ]:
TARGET = "modal_price"

## 6. Select initial features

In [ ]:
FEATURES = [
    "min_price",
    "max_price",
    "average_price",
    "price_range",
    "year",
    "month",
    "day",
    "day_of_week"
]

missing_features = [
    feature for feature in FEATURES
    if feature not in df.columns
]

print("Missing features:", missing_features)

## 7. Remove rows with missing ML values

In [ ]:
model_df = df[FEATURES + [TARGET]].dropna()

print("Data available for modeling:", model_df.shape)

## 8. Time-based train/test split
80% oldest observations → train, 20% newest → test. No random `train_test_split` here.

In [ ]:
split_index = int(len(model_df) * 0.8)

train = model_df.iloc[:split_index]
test = model_df.iloc[split_index:]

print("Training samples:", len(train))
print("Testing samples:", len(test))

## 9. Separate X and y

In [ ]:
X_train = train[FEATURES]
y_train = train[TARGET]

X_test = test[FEATURES]
y_test = test[TARGET]

print("X_train:", X_train.shape)
print("y_train:", y_train.shape)

print("X_test:", X_test.shape)
print("y_test:", y_test.shape)

## 10. Build the baseline model
Linear Regression, before XGBoost.

In [ ]:
from sklearn.linear_model import LinearRegression

baseline_model = LinearRegression()

baseline_model.fit(
    X_train,
    y_train
)

## 11. Generate predictions

In [ ]:
baseline_predictions = baseline_model.predict(X_test)

print(baseline_predictions[:10])

## 12. Evaluate the baseline

In [ ]:
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

mae = mean_absolute_error(
    y_test,
    baseline_predictions
)

rmse = np.sqrt(
    mean_squared_error(
        y_test,
        baseline_predictions
    )
)

r2 = r2_score(
    y_test,
    baseline_predictions
)

mape = np.mean(
    np.abs(
        (y_test - baseline_predictions)
        / y_test
    )
) * 100

print("Baseline Model Performance")
print("--------------------------")
print(f"MAE  : {mae:.2f}")
print(f"RMSE : {rmse:.2f}")
print(f"MAPE : {mape:.2f}%")
print(f"R²   : {r2:.4f}")

## 13. Visualize actual vs predicted

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(12, 5))

plt.plot(
    y_test.values,
    label="Actual"
)

plt.plot(
    baseline_predictions,
    label="Predicted"
)

plt.xlabel("Test Samples")
plt.ylabel("Modal Price")
plt.title("Actual vs Predicted Agricultural Price")

plt.legend()
plt.tight_layout()
plt.show()

## 14. Create a prediction table

In [ ]:
results = pd.DataFrame({
    "actual_price": y_test.values,
    "predicted_price": baseline_predictions
})

results["error"] = (
    results["actual_price"]
    - results["predicted_price"]
)

results.head(10)

## 15. Save the baseline results

In [ ]:
import os

os.makedirs("../data/processed", exist_ok=True)

results.to_csv(
    "../data/processed/baseline_predictions.csv",
    index=False
)

print("Baseline predictions saved.")

## 16. Save the baseline model

In [ ]:
import os
import joblib

os.makedirs("../models", exist_ok=True)

joblib.dump(
    baseline_model,
    "../models/baseline_linear_regression.pkl"
)

print("Baseline model saved.")

## Next step
Once these metrics are in hand, Task 5 swaps `LinearRegression` for `XGBRegressor` on the same train/test split and features, and compares MAE/RMSE/MAPE/R² against this baseline.